# Задача 2 · Дообучение модели поиска для классификатора по ПП РФ № 1875
**Вход:** наименование товара · **Выход:** код ОКПД2 → позиция ПП 1875 (правилом) → вердикт и уверенность.

Дообучаем эмбеддер на парах «наименование из закупки → код ОКПД2». Модель выбирается переключателем `MODEL`:
`e5-large` — та же, что сейчас в бэкенде `b2b-center`; `frida` — маленькая `rubert-mini-frida`, примерно в 6 раз быстрее.
Индекс поиска — как в бэкенде: наименования кодов + синонимы, плюс **формулировки от LLM (doc2query)** для каждого кода.
В конце — упаковка в **ONNX int8** с замером скорости и сравнением с PyTorch.
Эпоха выбирается по `dev.csv` (реальные закупки). `holdout.csv` сюда **не загружать** — он для финального замера.

Kaggle: **GPU T4 ×1**, **Internet On**, датасет `pp1875-data` (папка `kaggle_dataset/` из архива).

In [ ]:
# Библиотеки — те, что уже стоят в образе Kaggle. Версии НЕ закрепляем: старые transformers/sentence-transformers
# несовместимы со свежим PyTorch образа (Python 3.13) и ломают импорт. Сохранённая модель — обычный формат
# sentence-transformers (safetensors + конфиги), бэкенд загружает его своей версией библиотеки.
import os, re, glob, json, time, copy, random, shutil, importlib, sys, subprocess
import importlib.util
if importlib.util.find_spec("sentence_transformers") is None:            # в образе Kaggle обычно уже есть
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

print(f"PyTorch: {torch.__version__} | transformers: {transformers.__version__} | "
      f"sentence-transformers: {sentence_transformers.__version__}")
print(f"Python: {sys.version.split()[0]} | CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

In [ ]:
# Константы
DATA_DIR = "/kaggle/input/pp1875-data"          # Kaggle Dataset (содержимое kaggle_dataset/)
OUT_DIR = "/kaggle/working/artifacts"            # модель и файлы для бэкенда

OKPD2_FILE = "okpd2.csv"                         # из b2b-center: code;title;parent_code;is_leaf
SYNONYMS_FILE = "okpd2_synonyms.csv"             # из b2b-center: code;text — бытовые названия
PP_FILE = "pp1875.csv"                           # сверенные позиции ПП 1875, ред. 30.09.2026
TRAIN_FILES = ["train_v2.csv", "train_pairs.csv", "ktru.csv", "synth_weak.csv"]  # v2: КТРУ + синтетика
#   GigaChat + искажения как в ЕИС + наименования; необязательные (если есть в датасете): ktru.csv — полный КТРУ
#   (выгрузка из ЕИС), synth_weak.csv — синтетика по слабым темам (gigachat_generate.py --prefixes)
#                (text, okpd2, negative_code, source); train_pairs.csv — ради ~1000 пар КТРУ с кодом-группой (.000),
#                которых нет в v2; совпадения убираются. Старая схема — только ["train_pairs.csv"]
VAL_FILE = "dev.csv"                             # реальные закупки — по ним выбирается эпоха
TEST_FILES = ["testset.csv", "stress.csv", "validation.csv"]   # только для отчёта; holdout.csv не класть!
FIXES_FILE = "label_fixes.csv"                   # исправленные ошибки заказчиков в эталонах (как в eval.py)
PP_EDITION, PP_URL = "30.09.2026", ""

MODEL = "e5-large"                               # "e5-large" (как в бэкенде) | "frida" (маленькая, ~×6 быстрее)
PRESETS = {                                      # имя экспорта должно содержать «e5»/«frida»: по нему бэкенд ставит префиксы
    "e5-large": dict(BI_MODEL="intfloat/multilingual-e5-large", Q_PREFIX="query: ", D_PREFIX="passage: ",
                     EXPORT_NAME="e5-large-pp1875", BI_EPOCHS=2, BI_BS=64, BI_LR=1e-5, GRAD_CKPT=True),
    "frida":    dict(BI_MODEL="sergeyzh/rubert-mini-frida", Q_PREFIX="search_query: ", D_PREFIX="search_document: ",
                     EXPORT_NAME="rubert-mini-frida-pp1875", BI_EPOCHS=4, BI_BS=128, BI_LR=5e-5, GRAD_CKPT=False),
}
globals().update(PRESETS[MODEL])                 # BI_MODEL, префиксы, имя экспорта, эпохи, батч, lr
HF_REPO = ""                                     # напр. "BOBUS2317/e5-large-pp1875-onnx": выгрузить ONNX на Hugging Face
#                                                  (токен — Kaggle Secrets → HF_TOKEN); бэкенд: EMBED_MODEL=<HF_REPO>
HF_PRIVATE = False                               # публичная — чтобы docker-сборка у жюри скачала без токена

COMPARE_MODELS = {                               # zero-shot сравнение ({} — пропустить)
    "intfloat/multilingual-e5-large": ("query: ", "passage: "),
    "deepvk/USER-bge-m3": ("", ""),
    "sergeyzh/rubert-mini-frida": ("search_query: ", "search_document: "),
}

MAX_LEN, TOPK = 64, 20                           # наименования короткие; бэкенд берёт top-20 (retrieve_k)
BI_SCALE = 20.0                                  # эпохи, батч и lr — в PRESETS
OVERFETCH = 10                                   # у кода несколько текстов -> берём top-K×10 текстов и сворачиваем по коду
HARD_NEG_POOL = 15                               # трудный негатив — из top-15 похожих кодов

TRAIN_CE = False                                 # cross-encoder: в бэкенде убран (ухудшал точность), по умолчанию выкл.
CE_MODEL = "DiTy/cross-encoder-russian-msmarco"
RERANK_K, CE_EPOCHS, CE_GROUPS, CE_NEG, CE_LR, CE_MAX_LEN, CE_MAX_TRAIN = 10, 2, 8, 7, 2e-5, 160, 40000

MAX_PER_CODE = 40                                # не больше 40 пар КТРУ на код (КТРУ перекошен в медицину)
SELF_PAIRS = True                                # «наименование кода -> код»: все коды хоть раз в обучении
AUGMENT_FRAC, VAL_FRAC, VAL_MAX = 0.0, 0.1, 3000   # 0: искажения уже в train_v2 (для старой выборки — 0.3)
V2_NEG_SHARE = 0.5                               # доля примеров с трудным негативом из выборки (соседний код, часто
#                                                  с другим режимом ПП); остальным негатив подбирает модель

# Тексты от GigaChat (tools/gigachat_generate.py на своём ПК) — файлы в датасете. Сам ноутбук LLM не запускает.
USE_SYNTH = False                                # синтетика GigaChat уже внутри train_v2 (для старой выборки — True)
SYNTH_CACHE = "synth.csv"                        # из gigachat_out/
SYNTH_MAX_CODES = 6500                           # тот же отбор кодов, что в скрипте GigaChat — для проверки полноты
DOC2Q = True                                     # формулировки заказчиков для каждого кода -> доп. тексты индекса
DOC2Q_CACHE = "okpd2_generated_raw.csv"          # из gigachat_out/
DOC2Q_PER_CODE = 6
DOC2Q_FILTER_K = 50                              # формулировка остаётся, если исходная модель находит её код в top-50

EXPORT_ONNX = True                               # в архив идёт только ONNX int8 (PyTorch-модель — промежуточный файл)
ONNX_QCONFIG = "avx2"                            # "avx2" — любой x86-сервер (как модель по умолчанию в бэкенде);
#                                                  "avx512_vnni" — быстрее на новых Xeon; "arm64" — Mac M*, ARM-серверы
ONNX_EVAL_POOL = 50                              # оценка int8: переранжирование top-50 кодов fp32-индекса

CPU_THREADS, SEED = 4, 42

globals().update(json.loads(os.environ.get("PP_CFG_JSON", "{}")))   # переопределение для локальных тестов
if HF_REPO and not any(k in HF_REPO.lower() for k in ("e5", "frida")):
    raise ValueError("В HF_REPO должно быть «e5» или «frida»: по имени модели бэкенд выбирает префиксы запроса")

# Папка датасета: если имя отличается от pp1875-data (или Kaggle смонтировал его глубже) — ищем по файлам
NEED = ["okpd2.csv", "pp1875.csv", "pp1875_core.py"]
if not all(glob.glob(os.path.join(DATA_DIR, "**", f), recursive=True) for f in NEED):
    found = sorted({os.path.dirname(f) for f in glob.glob("/kaggle/input/**/okpd2.csv", recursive=True)})
    if found:
        DATA_DIR = found[0]
print("Данные:", DATA_DIR)
for f in NEED + ["onnx_export.py", "synth.csv", "okpd2_generated_raw.csv"]:     # одна копия каждого файла
    copies = sorted(glob.glob(f"/kaggle/input/**/{f}", recursive=True))
    if len(copies) > 1:
        used = (sorted(glob.glob(os.path.join(DATA_DIR, "**", f), recursive=True)) or copies)[0]
        print(f"!!! {f}: {len(copies)} копии в датасете, используется {used}. "
              "Оставьте одну, иначе можно взять старую версию:", copies)
missing = [f for f in NEED + (["onnx_export.py"] if EXPORT_ONNX else [])
           if not glob.glob(os.path.join(DATA_DIR, "**", f), recursive=True)]
if missing:
    print("Содержимое /kaggle/input:", sorted(glob.glob("/kaggle/input/**", recursive=True))[:40])
    raise FileNotFoundError(f"В датасете нет файлов {missing}. Загрузите их в датасет (New Version) и подключите "
                            "его через Add Input (onnx_export.py нужен только при EXPORT_ONNX = True)")
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = random.Random(SEED)

In [ ]:
# Чтение файлов и ядро (поиск + правило «код ОКПД2 -> позиция ПП»)
def path(p):
    """Файл в DATA_DIR; если датасет загружен архивом с вложенной папкой — ищем глубже."""
    if os.path.isabs(p):
        return p
    direct = os.path.join(DATA_DIR, p)
    if os.path.exists(direct):
        return direct
    found = sorted(glob.glob(os.path.join(DATA_DIR, "**", os.path.basename(p)), recursive=True)) or \
        sorted(glob.glob(os.path.join("/kaggle/input", "**", os.path.basename(p)), recursive=True))
    return found[0] if found else direct

def read_table(p):
    p = path(p)
    if p.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(p, dtype=str)           # dtype=str: иначе Excel превратит '26.20' в 26.2
    else:
        df = pd.read_csv(p, dtype=str, sep=None, engine="python", encoding="utf-8-sig", encoding_errors="replace")
    df.columns = [str(c).replace("﻿", "").strip().lower() for c in df.columns]
    return df

def pick(df, names, what):
    for n in names:
        if n in df.columns:
            return n
    raise ValueError(f"не найдена колонка {what}: ожидали одну из {names}, есть {list(df.columns)}")

TEXT_COLS = ["text", "name", "наименование", "наименование товара", "query", "товар"]
CODE_COLS = ["okpd2", "okpd2_expected", "code", "код", "код окпд2", "окпд2"]

if os.path.abspath(path("pp1875_core.py")) != os.path.abspath("pp1875_core.py"):
    shutil.copy(path("pp1875_core.py"), "pp1875_core.py")
sys.path.insert(0, os.getcwd())
import pp1875_core as core
importlib.reload(core)
from pp1875_core import (normalize_text, norm_code, build_docs, PPIndex, PP1875Classifier,
                         aggregate_outcomes, softmax, ece, fit_temperature, pick_threshold, NONE_KEY,
                         leaf_candidates, real_pair_counts, synth_targets)

def covers(parent, code):
    """То же правило иерархии, что в бэкенде (matcher.code_covers): префикс цифр + категория XX.XX.XX.XX0."""
    p, c = parent.replace(".", ""), code.replace(".", "")
    return c.startswith(p) or (len(p) == 9 and p.endswith("0") and len(c) == 9 and c[:8] == p[:8])

def verdict_of(key):
    """Вердикт по ключу позиций: 1 запрет < 2 ограничение < 3 преимущество; 0 — не подпадает."""
    return 0 if key in (None, NONE_KEY) else min(int(k[1:].split(".")[0]) for k in key.split("|"))

In [ ]:
# Справочники: ОКПД2 (документы для поиска — как в индексе бэкенда) и ПП 1875 (правила)
okpd_raw = read_table(OKPD2_FILE)
okpd = okpd_raw.rename(columns={pick(okpd_raw, ["code", "код", "okpd2"], "кода"): "code",
                                pick(okpd_raw, ["title", "name", "наименование"], "имени"): "name"})
if "is_leaf" in okpd.columns:                    # как в бэкенде: только коды нижнего уровня, чистые наименования
    leaf = okpd[okpd["is_leaf"].astype(str) == "1"]
    docs = pd.DataFrame({"code": leaf["code"].map(norm_code), "name": leaf["name"], "doc_text": leaf["name"]})
    docs = docs.dropna(subset=["code"]).drop_duplicates("code").reset_index(drop=True)
else:
    docs = build_docs(okpd, ctx_levels=0)
DOC_TEXT = dict(zip(docs["code"], docs["doc_text"]))
LEAVES = set(docs["code"])

pp_df = read_table(PP_FILE)
pp = PPIndex(pp_df, PP_URL, PP_EDITION)
docs["outcome"] = docs["code"].map(pp.outcome)

print(f"ОКПД2: {len(okpd)} кодов -> {len(docs)} кодов нижнего уровня (документы для поиска)")
print(f"ПП 1875: {len(pp.df)} позиций, приложения: {pp.df['appendix'].value_counts().to_dict()}")
print(f"Кодов нижнего уровня под позициями ПП: {(docs['outcome'] != NONE_KEY).sum()}")
display(docs.sample(min(5, len(docs)), random_state=0))

def gold_verdicts(code):
    """Эталонные вердикты кода (для группы — все вердикты её кодов, как в eval.py бэкенда)."""
    codes = [code] if code in LEAVES else [l for l in LEAVES if covers(code, l)] or [code]
    return {verdict_of(pp.outcome(c)) for c in codes}

In [ ]:
# Обучающие пары (КТРУ + синонимы), dev и тестовые наборы
def load_pairs(files, source=None):
    out = []
    for f in files:
        if not os.path.exists(path(f)):
            print("нет файла", f); continue
        df = read_table(f)
        t, c = pick(df, TEXT_COLS, "текста"), pick(df, CODE_COLS, "кода")
        src = df["source"] if "source" in df.columns and source is None else (source or os.path.basename(f))
        neg = df["negative_code"].map(norm_code) if "negative_code" in df.columns else None
        out.append(pd.DataFrame({"text": df[t], "code": df[c].map(norm_code), "source": src, "neg_code": neg}))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["text", "code", "source"])

fixes = {}
if os.path.exists(path(FIXES_FILE)):
    fx = read_table(FIXES_FILE)
    fixes = dict(zip(fx["name"], fx["okpd2_fixed"].map(norm_code)))

def load_eval(f):
    df = read_table(f)
    names = df[pick(df, TEXT_COLS, "текста")]
    codes = [fixes.get(n, norm_code(c)) for n, c in zip(names, df[pick(df, CODE_COLS, "кода")])]
    t = pd.DataFrame({"raw": names, "text": names.map(normalize_text), "code": codes})
    t["gold"] = t["code"].map(lambda c: pp.outcome(c) if c else None)
    t["category"] = df["category"] if "category" in df.columns else os.path.splitext(os.path.basename(f))[0]
    return t.dropna(subset=["code"]).reset_index(drop=True)

real = load_pairs(TRAIN_FILES)
syn = load_pairs([SYNONYMS_FILE], source="synonym").dropna(subset=["code", "text"])
real = pd.concat([real, syn], ignore_index=True)

# индекс поиска как в бэкенде (seed.okpd2_texts): наименования кодов нижнего уровня + синонимы к ним
INDEX_BASE = pd.concat([pd.DataFrame({"code": docs["code"], "text": docs["doc_text"], "kind": "official"}),
                        syn.loc[syn["code"].isin(LEAVES), ["code", "text"]].assign(kind="synonym")],
                       ignore_index=True).drop_duplicates(["code", "text"]).reset_index(drop=True)
INDEX = INDEX_BASE                               # после doc2query сюда добавятся сгенерированные формулировки
print(f"Индекс поиска: {len(INDEX_BASE)} текстов (наименования + синонимы)")
real["text"] = real["text"].map(normalize_text)
real = real.dropna(subset=["code", "text"])
real = real[real["text"].str.len() > 2].drop_duplicates(["text", "code"])
n0 = len(real)
REAL_COUNTS = real_pair_counts(real, LEAVES)                           # для отбора кодов синтетики (как в gigachat-скрипте)

# Один и тот же текст у разных кодов (78 синонимов вроде «бинт», «ip телефон») — противоречивые примеры: из обучения
# убираем (в индексе поиска они остаются, как в бэкенде)
amb = real.groupby(real["text"].str.lower())["code"].nunique()
real = real[~real["text"].str.lower().isin(set(amb[amb > 1].index))]

# Коды-группы КТРУ (10.12.10.000 = группа 10.12.10, коды старой редакции) — 10% пар. Раньше они молча
# выбрасывались; теперь код нижнего уровня внутри группы выберет исходная модель (ячейка «Модель»)
GROUP_CANDS = {c: leaf_candidates(c, sorted(LEAVES)) for c in set(real["code"]) - LEAVES}
group_pairs = real[real["code"].map(lambda c: c not in LEAVES and len(GROUP_CANDS.get(c, [])) > 0)].copy()
n_lost = int((~real["code"].isin(LEAVES)).sum() - len(group_pairs))
real = real[real["code"].isin(LEAVES)]
is_ktru = real["source"].isin(["train_pairs.csv", "v2_ktru", "ktru"])          # лимит — только на КТРУ (перекошен в медицину)
ktru, other = real[is_ktru], real[~is_ktru]
real = pd.concat([ktru.sample(frac=1, random_state=SEED).groupby("code").head(MAX_PER_CODE), other],
                 ignore_index=True)
print(f"Пары КТРУ + синонимы: {n0} | неоднозначных текстов убрано: {int((amb > 1).sum())} | "
      f"коды-группы: {len(group_pairs)} пар (+{n_lost} без подходящей группы) | итого {len(real)} "
      f"(КТРУ ≤{MAX_PER_CODE} на код), кодов: {real['code'].nunique()}")

dev = load_eval(VAL_FILE) if os.path.exists(path(VAL_FILE)) else None
test = pd.concat([load_eval(f) for f in TEST_FILES if os.path.exists(path(f))], ignore_index=True) \
    if any(os.path.exists(path(f)) for f in TEST_FILES) else None
held = set((dev["text"] if dev is not None else pd.Series(dtype=str)).tolist() +
           (test["text"] if test is not None else pd.Series(dtype=str)).tolist())
held_low = {t.lower() for t in held}
is_held = lambda s: s.str.lower().isin(held_low)                        # без учёта регистра: «Ноутбук» = «ноутбук»
n_leak = int(is_held(real["text"]).sum() + is_held(group_pairs["text"]).sum())
real, group_pairs = real[~is_held(real["text"])], group_pairs[~is_held(group_pairs["text"])]   # dev и тест не в обучении
print(f"Убрано из обучения совпадений с dev/тестом: {n_leak}")
print(f"Dev: {0 if dev is None else len(dev)} товаров | Тест:",
      {} if test is None else test["category"].value_counts().to_dict())

In [ ]:
# Синтетика от GigaChat: наименования в стиле закупок для кодов ПП и их «соседей» вне ПП (tools/gigachat_generate.py)
def load_texts(name, what, flag):
    """Готовые тексты от GigaChat из датасета (text, code, source); без файла — остановка с подсказкой."""
    if not name or not os.path.exists(path(name)):
        raise FileNotFoundError(
            f"Нет файла {what} «{name}» в датасете. Сгенерируйте его скриптом tools/gigachat_generate.py "
            f"(README, раздел 6а), добавьте в датасет и укажите имя в константах. Отключить: {flag} = False в константах")
    df = pd.read_csv(path(name), dtype=str)
    empty = int(df["text"].isna().sum())                               # коды, на которых GigaChat ничего не вернул
    return df.dropna(subset=["text", "code"])[["text", "code", "source"]].reset_index(drop=True), empty

synth = pd.DataFrame(columns=["text", "code", "source"])
if USE_SYNTH:
    synth, n_empty = load_texts(SYNTH_CACHE, "синтетики", "USE_SYNTH")
    need = set(synth_targets(docs, REAL_COUNTS, SYNTH_MAX_CODES, SEED))
    miss = len(need - set(synth["code"]))
    print(f"Синтетика GigaChat: {len(synth)} строк, {synth['code'].nunique()} кодов | пустых ответов {n_empty} | "
          f"кодов из отбора без строк: {miss} из {len(need)}")
    if miss > 0.05 * len(need):
        print("!!! Синтетика неполная или сделана старой версией скрипта — допишите её tools/gigachat_generate.py")
    display(synth.sample(min(10, len(synth)), random_state=0))

In [ ]:
# doc2query: формулировки заказчиков от GigaChat для каждого кода -> дополнительные тексты индекса
# Модель от этого не меняется: запрос «ноутбук 15.6 i5» совпадает уже не с «Компьютеры портативные массой...»,
# а с похожей формулировкой. Включаются в экспорт, только если не ухудшают dev и тест (ячейка «Обучение»).
generated = pd.DataFrame(columns=["text", "code", "source"])
if DOC2Q:
    generated, n_empty = load_texts(DOC2Q_CACHE, "формулировок doc2query", "DOC2Q")
    print(f"Формулировки GigaChat: {len(generated)} строк, {generated['code'].nunique()} из {len(docs)} кодов | "
          f"пустых ответов {n_empty}")
    display(generated.sample(min(10, len(generated)), random_state=0))

In [ ]:
# Сборка выборки: КТРУ + синонимы + синтетика + наименования кодов, аугментация; val = dev.csv
synth["text"] = synth["text"].map(normalize_text)
synth = synth[synth["code"].isin(LEAVES) & ~is_held(synth["text"])]

def split(df, frac):
    idx = np.random.permutation(len(df)); k = int(len(df) * frac)
    return df.iloc[idx[k:]], df.iloc[idx[:k]]

if dev is not None and len(dev):
    train, val = pd.concat([real, synth], ignore_index=True), dev.copy()
    print("Валидация: dev.csv (реальные закупки)")
else:                                                                   # запасной вариант без dev.csv
    r_tr, r_va = split(real, VAL_FRAC); s_tr, s_va = split(synth, VAL_FRAC)
    train, val = pd.concat([r_tr, s_tr], ignore_index=True), pd.concat([r_va, s_va]).head(VAL_MAX).reset_index(drop=True)
    val["gold"] = val["code"].map(pp.outcome)
    print("Валидация: 10% случайных пар (dev.csv не найден)")

TAILS = [", шт", " - 10 шт.", ", упаковка", " (или эквивалент)", ", ГОСТ", ", в ассортименте"]
def augment(t):                                   # шум как в реальных спецификациях
    r, w = rng.random(), t.split()
    if r < 0.3 and "," in t:  return t.split(",")[0]                      # только «голое» наименование
    if r < 0.55 and len(t) > 6:                                           # опечатка
        i = rng.randrange(1, len(t) - 2); return t[:i] + t[i + 1] + t[i] + t[i + 2:]
    if r < 0.8:  return t + rng.choice(TAILS)                             # хвост из спецификации
    if len(w) > 3: w.pop(rng.randrange(len(w))); return " ".join(w)       # пропуск слова
    return t

aug = train.sample(frac=AUGMENT_FRAC, random_state=1).copy()
aug["text"] = aug["text"].map(augment).map(normalize_text); aug["source"] += "+aug"
# «наименование кода -> код» — только для кодов без других примеров (чтобы каждый код хоть раз был в обучении,
# но 15 тыс. тривиальных пар не размывали батчи)
uncovered = docs[~docs["code"].isin(set(train["code"]) | set(group_pairs["code"].map(lambda c: GROUP_CANDS[c][0]
                                                                                  if len(GROUP_CANDS[c]) == 1 else "")))]
parts = [train, aug] + ([pd.DataFrame({"text": uncovered["name"].map(normalize_text), "code": uncovered["code"],
                                       "source": "okpd2_name"})] if SELF_PAIRS else [])
train_all = pd.concat(parts, ignore_index=True).drop_duplicates(["text", "code"]).sample(frac=1, random_state=2).reset_index(drop=True)

print(train_all["source"].value_counts().to_string())
print(f"Train: {len(train_all)} | Val: {len(val)}")

In [ ]:
# Оценка: код top-1 / top-K (правило иерархии как в бэкенде), вердикт ПП (как eval.py), точные позиции ПП
def make_clf(bi, emb, ce=None, ce_tok=None, texts=None, dev_=None, **calib):
    base = dict(q_prefix=Q_PREFIX, d_prefix=D_PREFIX, topk=TOPK, rerank_k=RERANK_K, overfetch=OVERFETCH,
                ce_max_len=CE_MAX_LEN, pp_edition=PP_EDITION, pp_url=PP_URL)
    base.update(calib)
    return PP1875Classifier(bi, docs, emb, pp, ce, ce_tok, base, dev_ or str(device),
                            INDEX if texts is None else texts)

def index_docs(model, qp=None, dp=None, texts=None):
    """Эмбеддинги всех текстов индекса (по умолчанию INDEX), fp16 на GPU."""
    clf = make_clf(model, None, texts=texts, **({"q_prefix": qp, "d_prefix": dp} if qp is not None else {}))
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        clf.doc_emb = clf.encode_docs(clf.texts["text"].tolist(), 128)
    return clf

def evaluate(clf, df, use_ce=False, T=None):
    codes, logits, tname = clf.candidates(df["text"].tolist(), use_ce)
    T = T or clf.calib[tname]
    rows = []
    for (cc, lg), g, gold in zip(zip(codes, logits), df["code"], df["gold"]):
        p = softmax(lg, T); key, conf = aggregate_outcomes(cc, p, pp)[0]
        best = cc[int(np.argmax(p))]
        rows.append({"pred_code": best, "pred_pp": key, "conf": conf,
                     "code_top1": covers(g, best), "code_topk": any(covers(g, c) for c in cc),
                     "verdict_ok": verdict_of(pp.outcome(best)) in gold_verdicts(g),   # как вердикт в eval.py
                     "pp_ok": key == gold})
    out = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    m = {"code_top1": out.code_top1.mean(), f"code_top{TOPK}": out.code_topk.mean(),
         "verdict": out.verdict_ok.mean(), "pp_exact": out.pp_ok.mean(), "ece": ece(out.conf, out.pp_ok)}
    return {k: round(float(v), 4) for k, v in m.items()}, out, (codes, logits)

def eval_sets(clf, use_ce=False):
    mv, _, _ = evaluate(clf, val, use_ce)
    mt = evaluate(clf, test, use_ce)[0] if test is not None and len(test) else {}
    return mv, mt

def fmt(m, pref):
    return " | ".join(f"{pref} {k}: {v:.4f}" for k, v in m.items() if k in ("code_top1", "verdict"))

In [ ]:
# Zero-shot сравнение эмбеддеров без обучения (тот же выбор, что в reports/models.md бэкенда)
cmp = []
for name, (qp, dp) in COMPARE_MODELS.items():
    try:
        m = SentenceTransformer(name, device=str(device)); m.max_seq_length = MAX_LEN
        t0 = time.time(); c = index_docs(m, qp, dp); te = time.time() - t0
        mv, mt = eval_sets(c)
        cmp.append({"model": name, **{f"val_{k}": v for k, v in mv.items()}, "test_verdict": mt.get("verdict"),
                    "index_s": round(te, 1)})
        print(f"{name:35s} | {fmt(mv, 'Val')} | Test verdict: {mt.get('verdict', 0):.4f}", flush=True)
        del m, c; torch.cuda.empty_cache()
    except Exception as e:
        print(name, "ошибка:", e)
if cmp: display(pd.DataFrame(cmp).sort_values("val_verdict", ascending=False))

In [ ]:
# Bi-encoder: модель, трудные негативы, DataLoader
# Повторный запуск ячейки в той же сессии: освободить видеопамять от прошлой модели, оптимизатора и LLM
import gc
for _n in ("bi", "bi_opt", "bi_sched", "scaler", "best_state", "clf0", "clf0_base", "clf_bi", "clf", "cpu_clf",
           "bi_cpu", "ce", "ce_opt", "check", "loaded", "m"):
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"Видеопамять: свободно {free / 2**30:.1f} из {total / 2**30:.1f} ГБ")
    if free < 6 * 2**30:
        raise RuntimeError("Видеопамять занята прошлыми запусками в этой сессии. Run → Factory reset, затем Run All")

bi = SentenceTransformer(BI_MODEL, device=str(device)); bi.max_seq_length = MAX_LEN
clf0_base = index_docs(bi, texts=INDEX_BASE)                  # индекс без doc2query — для сравнения
nogen_val, nogen_test = eval_sets(clf0_base)

# doc2query: чистка формулировок -> индекс. Остаются те, у которых исходная модель находит их код в top-50
# (отсекает ошибки LLM), без совпадений с dev/тестом (утечка) и без текстов, которые LLM дала разным кодам.
if len(generated):
    g = generated.dropna(subset=["text", "code"]).copy()
    g["text"] = g["text"].map(normalize_text)
    g = g[g["code"].isin(LEAVES) & (g["text"].str.len() > 2) & ~g["text"].isin(held)]
    g = g.drop_duplicates(["text", "code"])
    amb = g.groupby(g["text"].str.lower())["code"].nunique()
    g = g[~g["text"].str.lower().isin(set(amb[amb > 1].index))]
    clf0_base.calib["topk"] = DOC2Q_FILTER_K
    print(f"doc2query: проверка {len(g)} формулировок исходной моделью (несколько минут, без индикатора)...", flush=True)
    t0 = time.time()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        cand = clf0_base.candidates(g["text"].tolist(), use_ce=False, batch_size=256)[0] if len(g) else []
    print(f"  готово за {time.time() - t0:.0f} c", flush=True)
    clf0_base.calib["topk"] = TOPK
    keep = [c in cc for c, cc in zip(g["code"], cand)]
    gen_index = g[keep]
    print(f"doc2query: {len(generated)} -> {len(g)} после чистки -> {len(gen_index)} прошли проверку top-{DOC2Q_FILTER_K} "
          f"({gen_index['code'].nunique()} кодов)")
    INDEX = pd.concat([INDEX_BASE, gen_index[["code", "text"]].assign(kind="generated")],
                      ignore_index=True).drop_duplicates(["code", "text"]).reset_index(drop=True)
    print(f"Индекс поиска: {len(INDEX)} текстов (было {len(INDEX_BASE)})")
else:
    gen_index = generated.iloc[0:0]

clf0 = index_docs(bi)
base_val, base_test = eval_sets(clf0)
print(f"До обучения, индекс без doc2query | Val: {nogen_val}\n                                  | Test: {nogen_test}")
print(f"До обучения, индекс с doc2query   | Val: {base_val}\n                                  | Test: {base_test}")

def mine_hard_negatives(clf, texts, gold, given=None):
    """Для каждого примера — похожий, но неверный код из top-HARD_NEG_POOL. given — негативы из выборки v2
    (соседний код той же группы); берутся с вероятностью V2_NEG_SHARE, если код допустим."""
    clf.calib["topk"] = HARD_NEG_POOL + 1
    codes, _, _ = clf.candidates(list(texts), use_ce=False, batch_size=256)
    clf.calib["topk"] = TOPK
    all_codes = docs["code"].tolist()
    given = given if given is not None else [None] * len(gold)
    out = []
    for cc, g, gn in zip(codes, gold, given):
        if isinstance(gn, str) and gn in DOC_TEXT and not covers(g, gn) and not covers(gn, g) and rng.random() < V2_NEG_SHARE:
            out.append(gn)
        else:
            out.append(rng.choice([c for c in cc if not covers(g, c) and not covers(c, g)] or all_codes))
    return out

# Коды-группы КТРУ: исходная модель выбирает код нижнего уровня внутри группы (ближайший по смыслу)
if len(group_pairs):
    gq = clf0.encode_queries(group_pairs["text"].tolist(), 256)
    cand_all = sorted({c for g in group_pairs["code"] for c in GROUP_CANDS[g]})
    cand_emb = dict(zip(cand_all, clf0.encode_docs([DOC_TEXT[c] for c in cand_all], 256)))
    picked = [cs[int(np.argmax([q @ cand_emb[c] for c in cs]))] if len(cs) > 1 else cs[0]
              for q, cs in zip(gq, (GROUP_CANDS[g] for g in group_pairs["code"]))]
    gp = pd.DataFrame({"text": group_pairs["text"].values, "code": picked, "source": "ktru_group"})
    gp = gp.groupby("code").head(MAX_PER_CODE)
    train_all = pd.concat([train_all, gp], ignore_index=True).drop_duplicates(["text", "code"]) \
        .sample(frac=1, random_state=3).reset_index(drop=True)
    print(f"Коды-группы КТРУ: {len(gp)} пар привязаны к {gp['code'].nunique()} кодам нижнего уровня | Train: {len(train_all)}")

t0 = time.time()
print(f"Поиск трудных негативов для {len(train_all)} примеров (несколько минут)...", flush=True)
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
    negs = mine_hard_negatives(clf0, train_all["text"].tolist(), train_all["code"].tolist(),
                               train_all["neg_code"].tolist() if "neg_code" in train_all.columns else None)
print(f"Трудные негативы: {len(negs)} за {time.time() - t0:.0f} c")

class TripletDataset(Dataset):
    def __init__(self, df, negs):
        self.texts, self.codes, self.negs = df["text"].tolist(), df["code"].tolist(), negs
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return self.texts[i], self.codes[i], self.negs[i]

bi_loader = DataLoader(TripletDataset(train_all, negs), batch_size=BI_BS, shuffle=True, drop_last=True,
                       collate_fn=lambda b: tuple(map(list, zip(*b))))

if GRAD_CKPT:
    bi[0].auto_model.gradient_checkpointing_enable()

def embed(model, texts):
    feats = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in model.tokenize(texts).items()}
    return F.normalize(model(feats)["sentence_embedding"], dim=-1)

bi_opt = torch.optim.AdamW(bi.parameters(), lr=BI_LR, weight_decay=0.01)
bi_steps = BI_EPOCHS * len(bi_loader)
bi_sched = get_linear_schedule_with_warmup(bi_opt, int(0.1 * bi_steps), bi_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

In [ ]:
# Цикл обучения bi-encoder: contrastive loss (in-batch + трудные негативы)
print("\n--- Старт обучения bi-encoder ---", flush=True)
# Эпоха выбирается по индексу БЕЗ doc2query (качество самой модели; про doc2query решается отдельно, после обучения).
# Оценка = вердикт + код@1 + 0,5·код@20: dev всего 88 товаров, и вердикт с код@1 часто дают ничью, а код@20 —
# плавный сигнал. При равенстве берётся обученная эпоха (раньше ничья оставляла исходную модель).
sel = lambda m: m["verdict"] + m["code_top1"] + 0.5 * m[f"code_top{TOPK}"]
history = [{"stage": "bi", "epoch": 0, "train_loss": None, **{f"val_{k}": v for k, v in nogen_val.items()},
            **{f"test_{k}": v for k, v in nogen_test.items()}}]
best_score, best_state, best_epoch = sel(nogen_val), None, 0

for epoch in range(BI_EPOCHS):
    bi.train()
    train_loss = 0.0

    pbar = tqdm(bi_loader, desc=f"Epoch {epoch+1}/{BI_EPOCHS} [Train]", leave=False)
    for anchors, codes, neg_codes in pbar:
        B = len(anchors)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            q = embed(bi, [Q_PREFIX + t for t in anchors])
            d = embed(bi, [D_PREFIX + DOC_TEXT[c] for c in codes + neg_codes])   # B позитивов + B негативов
            scores = (q @ d.T).float() * BI_SCALE

        # одинаковый код у другого примера в батче — не негатив, маскируем
        doc_codes = np.array(codes + neg_codes)
        same = torch.from_numpy(np.array(codes)[:, None] == doc_codes[None, :]).to(device)
        same[torch.arange(B), torch.arange(B)] = False
        loss = F.cross_entropy(scores.masked_fill(same, -1e4), torch.arange(B, device=device))

        bi_opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(bi_opt)
        torch.nn.utils.clip_grad_norm_(bi.parameters(), 1.0)
        scaler.step(bi_opt); scaler.update(); bi_sched.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / max(1, len(bi_loader))

    # Валидация: переиндексация ОКПД2 и метрики на dev / тесте
    bi.eval()
    mv, mt = eval_sets(index_docs(bi, texts=INDEX_BASE))
    history.append({"stage": "bi", "epoch": epoch + 1, "train_loss": avg_train_loss,
                    **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}})
    print(
        f"Epoch [{epoch+1:02d}/{BI_EPOCHS}] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Code@1: {mv['code_top1']:.4f} | "
        f"Val Code@{TOPK}: {mv[f'code_top{TOPK}']:.4f} | "
        f"Val Verdict: {mv['verdict']:.4f} | "
        f"Test Verdict: {mt.get('verdict', 0):.4f}",
        flush=True
    )
    score = sel(mv)
    print(f"  оценка выбора: {score:.4f} (лучшая {best_score:.4f}, эпоха {best_epoch})", flush=True)
    if score >= best_score:                                                 # ничья — в пользу обученной эпохи
        best_score, best_epoch = score, epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in bi.state_dict().items()}

if best_state is None:
    print(f"\n!!! Все эпохи хуже исходной модели на dev — экспортируется исходная {BI_MODEL}")
    bi = SentenceTransformer(BI_MODEL, device=str(device)); bi.max_seq_length = MAX_LEN
else:
    bi.load_state_dict(best_state)
    print(f"\nЭкспортируется эпоха {best_epoch} (оценка {best_score:.4f} против {sel(nogen_val):.4f} у исходной)")
bi.eval()
clf_bi = index_docs(bi)
bi_val, bi_test = eval_sets(clf_bi)
print(f"\nИтоговый bi-encoder | Val: {bi_val}\n                   | Test: {bi_test}")

# doc2query остаётся, только если с ним на dev не хуже (иначе экспортируется индекс без формулировок)
if len(gen_index):
    fin_nogen_val, fin_nogen_test = eval_sets(index_docs(bi, texts=INDEX_BASE))
    sc = lambda m: m.get("verdict", 0) + m.get("code_top1", 0)
    print(f"doc2query (вердикт + код@1): dev {sc(bi_val):.4f} с формулировками vs {sc(fin_nogen_val):.4f} без | "
          f"тест {sc(bi_test):.4f} vs {sc(fin_nogen_test):.4f}")
    # dev — всего 88 товаров, поэтому формулировки должны быть не хуже И на dev, И на тестовых наборах
    DOC2Q_USED = sc(bi_val) >= sc(fin_nogen_val) and sc(bi_test) >= sc(fin_nogen_test) - 0.01
    gen_val, gen_test = bi_val, bi_test
    if not DOC2Q_USED:
        print("!!! Формулировки doc2query ухудшили dev или тест — индекс экспортируется без них")
        INDEX = INDEX_BASE
        clf_bi = index_docs(bi)
        bi_val, bi_test = fin_nogen_val, fin_nogen_test
else:
    DOC2Q_USED = False

In [ ]:
# Cross-encoder (по умолчанию выключен: TRAIN_CE = False): группы «1 верный код + 7 трудных»
ce = ce_tok = None
if TRAIN_CE:
    ce_tok = AutoTokenizer.from_pretrained(CE_MODEL)
    ce = AutoModelForSequenceClassification.from_pretrained(CE_MODEL, num_labels=1, ignore_mismatched_sizes=True).to(device)

    ce_src = train_all[train_all["source"] != "okpd2_name"]
    ce_src = pd.concat([ce_src, train_all[train_all["source"] == "okpd2_name"].sample(frac=0.2, random_state=3)])
    ce_src = ce_src.sample(frac=1, random_state=4).head(CE_MAX_TRAIN)
    cand_codes, _, _ = clf_bi.candidates(ce_src["text"].tolist(), use_ce=False, batch_size=256)
    groups = []
    for t, g, cc in zip(ce_src["text"], ce_src["code"], cand_codes):
        ng = [c for c in cc[:RERANK_K + 5] if c != g][:CE_NEG]
        while len(ng) < CE_NEG:
            ng.append(rng.choice(docs["code"].tolist()))
        groups.append((t, [g] + ng))                                       # верный всегда на позиции 0

    ce_loader = DataLoader(groups, batch_size=CE_GROUPS, shuffle=True, collate_fn=lambda b: b)
    N = CE_NEG + 1
    ce_opt = torch.optim.AdamW(ce.parameters(), lr=CE_LR, weight_decay=0.01)
    ce_steps = CE_EPOCHS * len(ce_loader)
    ce_sched = get_linear_schedule_with_warmup(ce_opt, int(0.1 * ce_steps), ce_steps)
    print(f"Групп для cross-encoder: {len(groups)} | шагов: {ce_steps}")
else:
    print("Cross-encoder пропущен (TRAIN_CE = False)")

In [ ]:
# Цикл обучения cross-encoder: softmax по группе кандидатов
if TRAIN_CE:
    print("\n--- Старт обучения cross-encoder ---", flush=True)
    best_ce, best_ce_state = -1, None
    for epoch in range(CE_EPOCHS):
        ce.train()
        train_loss = 0.0
        pbar = tqdm(ce_loader, desc=f"Epoch {epoch+1}/{CE_EPOCHS} [Train]", leave=False)
        for batch in pbar:
            qs = [t for t, cs in batch for _ in cs]
            ds = [DOC_TEXT[c] for _, cs in batch for c in cs]
            enc = ce_tok(qs, ds, padding=True, truncation=True, max_length=CE_MAX_LEN, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = ce(**enc).logits[:, 0].view(len(batch), N)
            loss = F.cross_entropy(logits.float(), torch.zeros(len(batch), dtype=torch.long, device=device))
            ce_opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(ce_opt)
            torch.nn.utils.clip_grad_norm_(ce.parameters(), 1.0)
            scaler.step(ce_opt); scaler.update(); ce_sched.step()
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        ce.eval()
        mv, mt = eval_sets(make_clf(bi, clf_bi.doc_emb, ce, ce_tok), use_ce=True)
        history.append({"stage": "ce", "epoch": epoch + 1, "train_loss": train_loss / max(1, len(ce_loader)),
                        **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}})
        print(f"Epoch [{epoch+1:02d}/{CE_EPOCHS}] | Train Loss: {train_loss / max(1, len(ce_loader)):.4f} | "
              f"{fmt(mv, 'Val')} | Test Verdict: {mt.get('verdict', 0):.4f}", flush=True)
        if mv["verdict"] + mv["code_top1"] > best_ce:
            best_ce = mv["verdict"] + mv["code_top1"]
            best_ce_state = {k: v.detach().cpu().clone() for k, v in ce.state_dict().items()}
    ce.load_state_dict(best_ce_state); ce.eval()
    if best_ce <= bi_val["verdict"] + bi_val["code_top1"]:
        print("!!! cross-encoder не лучше одного bi-encoder на dev — не экспортируем"); ce = ce_tok = None

In [ ]:
# Калибровка уверенности (температура по ECE) — для ядра; в бэкенде своя калибровка (eval.py --fit-confidence)
use_ce = ce is not None
clf = make_clf(bi, clf_bi.doc_emb, ce, ce_tok)
_, _, (cc, lg) = evaluate(clf, val, use_ce=use_ce, T=1.0)
T, e = fit_temperature(lg, cc, val["gold"].tolist(), pp)
clf.calib.update({"T_ce" if use_ce else "T_bi": T})
final_val, val_df, _ = evaluate(clf, val, use_ce=use_ce)
tau = pick_threshold(val_df["conf"].values, val_df["pp_ok"].values, 0.95, min_support=10)
clf.calib["tau_auto"] = tau
print(f"T = {T:.3f} (ECE {e:.3f}) | порог auto = {tau:.2f}")

In [ ]:
# Итоги: до/после, тест по наборам, ошибки, графики обучения
base_ref = base_val if DOC2Q_USED else nogen_val                     # «до» — с тем же индексом, что ушёл в экспорт
res = pd.DataFrame([{"stage": f"{MODEL} без обучения", **base_ref}, {"stage": f"{MODEL} дообученная", **final_val}])
print("Dev (реальные закупки):"); display(res)
metrics = {"model": BI_MODEL, "preset": MODEL, "dev": res.to_dict("records"), "history": history, "T": T, "tau_auto": tau}

# Вклад doc2query: та же модель с индексом без сгенерированных формулировок и с ними
if len(gen_index):
    abl = pd.DataFrame([
        {"модель": "без обучения", "индекс": "наименования + синонимы", "val_code_top1": nogen_val["code_top1"],
         "val_verdict": nogen_val["verdict"], "test_verdict": nogen_test.get("verdict")},
        {"модель": "без обучения", "индекс": "+ doc2query", "val_code_top1": base_val["code_top1"],
         "val_verdict": base_val["verdict"], "test_verdict": base_test.get("verdict")},
        {"модель": "дообученная", "индекс": "наименования + синонимы", "val_code_top1": fin_nogen_val["code_top1"],
         "val_verdict": fin_nogen_val["verdict"], "test_verdict": fin_nogen_test.get("verdict")},
        {"модель": "дообученная", "индекс": "+ doc2query", "val_code_top1": gen_val["code_top1"],
         "val_verdict": gen_val["verdict"], "test_verdict": gen_test.get("verdict")},
    ])
    print(f"Вклад doc2query (без cross-encoder), в экспорт {'вошёл' if DOC2Q_USED else 'НЕ вошёл'}:"); display(abl)
    metrics["doc2query"] = {"used": bool(DOC2Q_USED), "texts": int(len(gen_index)), "table": abl.to_dict("records")}

if test is not None and len(test):
    rows = []
    for name_, c_ in (("до", clf0), ("после", clf)):
        _, tdf, _ = evaluate(c_, test, use_ce=use_ce and name_ == "после")
        g = tdf.groupby("category").agg(n=("verdict_ok", "size"), verdict=("verdict_ok", "mean"), code_top1=("code_top1", "mean"))
        rows.append(g.add_prefix(f"{name_}_"))
    by_cat = pd.concat(rows, axis=1)
    print("Тестовые наборы (вердикт и код до/после дообучения):"); display(by_cat.round(3))
    metrics["test_by_set"] = by_cat.reset_index().to_dict("records")
    print("Ошибки вердикта после дообучения:"); display(tdf[~tdf["verdict_ok"]][["category", "raw", "code", "pred_code", "pred_pp"]])

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hb = h[h["stage"] == "bi"]
ax[0].plot(hb["epoch"][1:], hb["train_loss"][1:], marker="o"); ax[0].set_title("Train loss")
for col, lab in (("val_verdict", "dev вердикт"), ("val_code_top1", "dev код top-1"), ("test_verdict", "тест вердикт")):
    if col in hb: ax[1].plot(hb["epoch"], hb[col], marker="o", label=lab)
ax[1].axhline(0.85, color="gray", lw=0.8, ls=":"); ax[1].set_title("Точность по эпохам (0 — без обучения)")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
ax[1].legend(); plt.show()

In [ ]:
# Скорость на CPU, как у жюри без видеокарты (критерий: 1 товар < 5 с, 100 товаров < 2 мин)
torch.set_num_threads(CPU_THREADS)
bi_cpu = copy.deepcopy(bi).to("cpu").float()
ce_cpu = copy.deepcopy(ce).to("cpu").float().eval() if ce is not None else None
cpu_clf = PP1875Classifier(bi_cpu, docs, clf.doc_emb, pp, ce_cpu, ce_tok, dict(clf.calib), "cpu", clf.texts)
bench = (val["text"].tolist() * 3)[:100]
cpu_clf.predict(bench[:2], use_ce=use_ce)                                     # прогрев
t0 = time.perf_counter(); cpu_clf.predict(bench[:1], use_ce=use_ce); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); cpu_clf.predict(bench, use_ce=use_ce); t100 = time.perf_counter() - t0
print(f"CPU ({CPU_THREADS} потока) | 1 товар: {t1:.2f} c | 100 товаров: {t100:.1f} c")

In [ ]:
# Экспорт: модель для бэкенда (+ опционально Hugging Face) и артефакты ядра
A = OUT_DIR
bi.save(f"{A}/{EXPORT_NAME}")                                               # папка модели для EMBED_MODEL бэкенда
if ce is not None:
    ce.save_pretrained(f"{A}/reranker"); ce_tok.save_pretrained(f"{A}/reranker")
docs[["code", "name", "doc_text"]].to_parquet(f"{A}/okpd2_docs.parquet", index=False)
np.save(f"{A}/doc_emb.npy", clf.doc_emb.astype(np.float16))                # эмбеддинги текстов индекса
clf.texts.to_parquet(f"{A}/okpd2_texts.parquet", index=False)              # тексты индекса: code, text, kind

# Файлы для data/ бэкенда: синонимы + формулировки doc2query (формат code;text) и готовый индекс
# (data/index/<slug модели>.npz — бэкенд возьмёт его вместо расчёта ~100 тыс. эмбеддингов на CPU)
def backend_model(onnx):
    """Значение EMBED_MODEL в бэкенде: репозиторий HF или папка в data/models/."""
    return HF_REPO or f"/data/models/{EXPORT_NAME}" + ("-onnx" if onnx else "")

def index_name(model, onnx):
    """Имя файла индекса в data/index/ — как seed.index_name бэкенда."""
    slug = re.sub(r"[^a-z0-9]+", "-", model.lower()).strip("-")
    return f"{slug}-onnx.npz" if onnx and not slug.endswith("onnx") else f"{slug}.npz"

BACKEND_EMBED_MODEL = backend_model(EXPORT_ONNX)
slug = index_name(BACKEND_EMBED_MODEL, EXPORT_ONNX)[:-4]
B = f"{A}/backend_data"; os.makedirs(f"{B}/index", exist_ok=True)
syn_raw = read_table(SYNONYMS_FILE)[["code", "text"]]
gen_out = (gen_index if DOC2Q_USED else gen_index.iloc[0:0])[["code", "text"]].sort_values(["code", "text"])
gen_out.to_csv(f"{B}/okpd2_generated.csv", sep=";", index=False)
pd.concat([syn_raw, gen_out]).drop_duplicates().sort_values(["code", "text"]) \
    .to_csv(f"{B}/okpd2_synonyms.csv", sep=";", index=False)
as_str = lambda col: np.array(clf.texts[col].astype(str).tolist(), dtype=str)   # не object: бэкенд читает allow_pickle=False
np.savez_compressed(f"{B}/index/{slug}.npz", codes=as_str("code"), texts=as_str("text"), kinds=as_str("kind"),
                    vectors=clf.doc_emb.astype(np.float16))
print(f"           data/: {len(gen_out)} формулировок doc2query, индекс index/{slug}.npz ({len(clf.texts)} текстов)")
pp_df.to_csv(f"{A}/pp1875.csv", index=False)
shutil.copy("pp1875_core.py", f"{A}/pp1875_core.py")
calib = {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in clf.calib.items()}
calib.update(bi_model=BI_MODEL, bi_dir=EXPORT_NAME, ce_model=CE_MODEL if ce is not None else "")
json.dump(calib, open(f"{A}/calib.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(metrics, open(f"{A}/metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)

if HF_REPO and not EXPORT_ONNX:                                           # с ONNX выгружается папка ONNX (последняя ячейка)
    from kaggle_secrets import UserSecretsClient
    url = bi.push_to_hub(HF_REPO, token=UserSecretsClient().get_secret("HF_TOKEN"), private=HF_PRIVATE)
    print(f"Модель выгружена на Hugging Face: {url}")

if not EXPORT_ONNX:                                                       # с ONNX архив собирается в последней ячейке
    shutil.make_archive(A.rstrip("/"), "zip", A)
    size_mb = os.path.getsize(A.rstrip("/") + ".zip") / (1024 * 1024)
    print(f"\nАртефакты сохранены в '{A}.zip' ({size_mb:.0f} MB); модель для бэкенда — папка '{EXPORT_NAME}'")
else:
    print(f"\nМодель PyTorch сохранена временно ('{EXPORT_NAME}') — в архив пойдёт только ONNX (последняя ячейка)")

In [ ]:
# Проверка: загрузка модели так, как это делает бэкенд (sentence-transformers + префиксы модели)
check = SentenceTransformer(f"{A}/{EXPORT_NAME}", device=str(device))   # GPU: 15 тыс. наименований на CPU — лишние минуты
qv = check.encode([Q_PREFIX + "Ноутбук 15.6\" Intel Core i5, 16 ГБ"], normalize_embeddings=True)
dv = check.encode([D_PREFIX + t for t in docs["doc_text"]], normalize_embeddings=True, batch_size=256)
top = np.argsort(-(qv @ dv.T)[0])[:3]
print("Ближайшие коды:", [(docs['code'][i], docs['doc_text'][i][:40]) for i in top])

loaded = PP1875Classifier.load(A, device="cpu")
for r in loaded.predict(["Ноутбук 15.6\" Intel Core i5, 16 ГБ ОЗУ, SSD 512 ГБ",
                         "Бумага для офисной техники А4 80 г/м2, 500 листов",
                         "Услуги по уборке помещений"], use_ce=ce is not None):
    pos = "; ".join(f"прил. {p['appendix']} поз. {p['item']} ({p['regime']})" for p in r["pp_positions"]) or "не входит"
    print(f"{r['query'][:45]:45s} | {r['confidence']:5.1f}% | ОКПД2 {r['okpd2']} | {pos}")
print("\nУспешная проверка модели и артефактов!")

In [ ]:
# ONNX + int8: упаковка модели для CPU, замер скорости, сравнение векторов и точности с PyTorch.
# Идёт ОТДЕЛЬНЫМ процессом (onnx_export.py) и последней ячейкой: установка optimum может сменить версию transformers,
# а загруженные в ноутбук библиотеки от этого ломаются. Модель и артефакты выше к этому моменту уже сохранены.
if EXPORT_ONNX:
    val[["text", "code"]].to_pickle(f"{OUT_DIR}/../onnx_val.pkl")
    (test[["text", "code", "category"]] if test is not None else val.iloc[0:0]).to_pickle(f"{OUT_DIR}/../onnx_test.pkl")
    shutil.copy(path("onnx_export.py"), "onnx_export.py")
    st_ver = sentence_transformers.__version__
    !pip install -q "sentence-transformers[onnx]=={st_ver}"
    # optimum подгружает diffusers, если он есть; diffusers из образа Kaggle несовместим с huggingface-hub,
    # который ставит optimum, и импорт падает. Для эмбеддера diffusers не нужен — удаляем.
    !pip uninstall -y -q diffusers
    !python onnx_export.py --art {OUT_DIR} --model-dir {EXPORT_NAME} --qconfig {ONNX_QCONFIG} --threads {CPU_THREADS} --max-len {MAX_LEN} --pool {ONNX_EVAL_POOL} --val {OUT_DIR}/../onnx_val.pkl --test {OUT_DIR}/../onnx_test.pkl

    # Архив только с ONNX: модель int8 + файлы для data/ бэкенда + метрики. PyTorch-модель (2,2 ГБ) и файлы ядра —
    # только если ONNX не получился (тогда это запасной вариант).
    onnx_dir = f"{OUT_DIR}/{EXPORT_NAME}-onnx"
    ok = bool(glob.glob(f"{onnx_dir}/onnx/model_*int8*.onnx"))
    if ok:
        q_files = sorted(os.path.basename(f) for f in glob.glob(f"{onnx_dir}/onnx/*.onnx"))
        print("ONNX-файлы:", q_files, "| внешние данные:", glob.glob(f"{onnx_dir}/onnx/*data*") or "нет")
        for f in os.listdir(OUT_DIR):
            if f not in (f"{EXPORT_NAME}-onnx", "backend_data", "metrics.json"):
                fp = os.path.join(OUT_DIR, f)
                shutil.rmtree(fp) if os.path.isdir(fp) else os.remove(fp)
    else:
        print("!!! ONNX не получился — в архив идёт модель PyTorch")
        npz_old = f"{OUT_DIR}/backend_data/index/{slug}.npz"               # индекс считался для варианта без ONNX?
        npz_new = f"{OUT_DIR}/backend_data/index/{index_name(f'/data/models/{EXPORT_NAME}', False)}"
        if os.path.exists(npz_old) and npz_old != npz_new:
            os.rename(npz_old, npz_new)
    shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)
    print(f"\nartifacts.zip ({os.path.getsize(OUT_DIR.rstrip('/') + '.zip') / 2**20:.0f} MB):", sorted(os.listdir(OUT_DIR)))
    if ok and HF_REPO:
        from huggingface_hub import HfApi
        from kaggle_secrets import UserSecretsClient
        api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
        api.create_repo(HF_REPO, private=HF_PRIVATE, exist_ok=True)
        api.upload_folder(repo_id=HF_REPO, folder_path=onnx_dir, commit_message=f"ONNX int8 ({ONNX_QCONFIG})")
        print(f"ONNX выгружен на Hugging Face: https://huggingface.co/{HF_REPO}")
    env = backend_model(True) if ok else f"/data/models/{EXPORT_NAME}"
    print("\nДля .env бэкенда:", f"EMBED_MODEL={env}", "EMBED_BACKEND=onnx" if ok else "EMBED_BACKEND=torch",
          f"EMBED_DIM={clf.doc_emb.shape[1]}" if clf.doc_emb.shape[1] != 1024 else "")
    print(f"Индекс: backend_data/index/{index_name(env, ok)} → data/index/;"
          " backend_data/okpd2_synonyms.csv → data/; затем app.seed --reindex и eval.py --fit-confidence")